## Modelling Estonian Energy Supply

Forecasting electricity prices in the Estonian grid is a challenging problem due to two compounding sources of complexity. First, supply-side uncertainty is high, as generation depends heavily on variable renewables such as wind, whose output is inherently difficult to predict. Second, Estonia operates within the interconnected Nordic-Baltic electricity market, meaning prices are strongly influenced by cross-border flows from neighboring bidding zones: Latvia, Lithuania, and Finland. Ignoring these spatial dependencies risks producing systematically biased forecasts.
These characteristics motivate a model that is both temporally aware and spatially structured. We represent the Nordic-Baltic grid as a graph, with bidding zones as nodes and physical cross-border transmission flows as edges, and apply a Spatio-Temporal Graph Neural Network (ST-GNN) ([Hadou et al., 2022](   https://doi.org/10.48550/arXiv.2110.02880)). ST-GNNs are well-suited to this setting, as they jointly learn spatial dependencies across graph nodes and temporal dynamics across time steps, offering greater flexibility than univariate time series models. However, this flexibility comes at a cost: ST-GNNs require substantially more data, are harder to interpret, and are sensitive to the quality of the graph structure provided. The model is trained on seven years of hourly data (2019–2025) from the [Elering API](https://dashboard.elering.ee/assets/swagger-ui/index.html), supplemented with weather data from [Open-Meteo](https://archive-api.open-meteo.com/v1/archive) and actual wind generation from [ENTSO-E](https://transparency.entsoe.eu).

### Data and features

The Elering API provided hourly production, renewable production, electricity prices, and cross-border import/export flows between Estonia and Finland, Latvia, Lithuania, and Russia (the latter disconnected from the Baltic grid in 2025). All data was originally recorded at 15-minute or hourly resolution and aligned to a common hourly frequency. The target variable is Estonian domestic energy production (MW).
Input features include current-timestep production, available energy (defined as production minus net exports to Finland and Latvia), renewable production, cross-border import/export flows to Finland and Latvia, electricity prices. All are measured at the current timestep and therefore fully observable at forecast time without leakage. Additionally, 24-hour lagged versions of production and available energy are included to capture daily periodicity. Weather variables (temperature and wind speed at 10 m) and cyclic calendar encodings (hour, day-of-week, month) complete the feature set. Actual metered wind generation from [ENTSO-E](https://transparency.entsoe.eu) is included as an explicit feature to give the model a direct view of the wind component within total production. Frequency deviation from 50 Hz is included as a [grid stability signal](https://www.neso.energy/energy-101/electricity-explained/how-do-we-balance-grid/what-frequency), reflecting real-time supply-demand imbalance across the synchronous grid .
No unnecessary lags are applied to current-state measurements such as production or flows, as these reflect the state of the system at forecast time and are fully observable. The 24-hour lag is retained only for features where yesterday's value at the same hour carries predictive signal beyond the current reading.


### Graph structure

The four bidding zones — Estonia (EE), Finland (FI), Latvia (LV), and Lithuania (LT) — form the nodes of the graph. Edges represent active cross-border interconnections, with Estonia connected to Finland and Latvia, and Latvia connected to Lithuania. All edges are undirected and weighted by the absolute mean historical import/export flow on each connection, normalised before training.
Estonia, as the target node, carries the full feature set described above. The neighboring nodes (FI, LV, LT) carry a reduced feature set consisting of their local electricity prices and the cross-border flows directed toward Estonia, reflecting the information that is physically transmitted across the interconnections. This asymmetry in node features is intentional: Estonia contains substantially more features than its neighbors, as the model is trained to predict Estonian domestic production.

<img src="../figures/Energy_grid.png" width="10%">
Figure 1: Graph structure of the Nordic-Baltic electricity market.



The topology reflects both the geographical layout of the region and the physical transmission infrastructure: edges correspond to active high-voltage interconnections between bidding zones. Estonia has no direct cable connection to Lithuania, so no edge is drawn between them. Russia, while historically connected to the Estonian grid, was excluded as a graph node due to limited data availability The electricity prices in Russia are not publicly transparent. However, the cross-border flow between Estonia and Russia is retained as a feature within the Estonian node itself.

### Sequence construction and splits

We use a sliding window of 48 hours (2 days) as the input sequence, with the model tasked to predict the 24-hour production profile of the following day. This sequence length yielded comparable performance to longer windows of 168 hours (1 week) at substantially lower computational cost.
Given that training a high-performing GNN was not the primary objective of this study, but rather a necessary step toward producing reliable energy simulations, a simple holdout strategy was adopted in place of full cross-validation. The last 20% of the data constitutes the validation set, and January 2026 is reserved entirely as the test set. Both partitions adhere to strict chronological ordering, with no shuffling across sets. A systematic cross-validation procedure would in principle allow for a more rigorous exploration of the trade-off between feature selection, hyperparameter tuning, and network architecture. However, given the computational and time constraints of this project, configurations were evaluated through structured manual experimentation, with the best-performing setup selected on the basis of validation performance.
All features and time series were differenced and normalised prior to training. Differencing was applied to address non-stationarity in the production signal, while normalisation constrains the input range across features, mitigating the risk of exploding gradients and promoting stable optimisation dynamics.
No unnecessary lags are applied to current-state measurements such as production or flows, as these quantities are fully observable at forecast time and introduce no leakage. The 24-hour lag is retained exclusively for features where the value at the equivalent hour on the preceding day carries predictive signal beyond the current observation.

### Model architecture
The paper by Hadou et al. (2022) introduces a Spatio-Temporal Graph Neural Network architecture designed to jointly process time-varying signals defined over a graph, by composing graph and time convolutional filters into a unified space-time shift operator. A key theoretical contribution is the proof that the architecture is stable under both graph perturbations (changes in network topology) and time perturbations (irregular sampling), meaning small changes in the underlying structure produce proportionally small changes in the output. The architecture is validated on decentralised control tasks such as multi-agent flocking and motion planning, where it outperforms conventional decentralised controllers.
Following the spatio-temporal forecasting framework of [Hadou et al. (2022)](https://doi.org/10.48550/arXiv.2110.02880), we implement a GATv2-based spatial module combined with a GRU temporal module.
The ST-GNN consists of two components: a spatial module and a temporal module. The spatial module applies two successive GATv2 graph attention layers to each hourly graph snapshot, with edge attributes encoding the normalised cross-border flow magnitudes. Each attention layer uses four attention heads with the outputs averaged, allowing the model to attend to different aspects of the neighbourhood structure simultaneously. Residual connections are added after each attention layer to stabilise training and prevent representational collapse in deeper networks.
The temporal module operates on the Estonia node exclusively. The Estonian node embedding is concatenated with an attention-weighted mean of the three neighbour embeddings (FI, LV, LT), where the attention weights are computed as a soft alignment between the Estonian embedding and each neighbour. Positional embeddings are added to the Estonian component to make the sequence order explicit, as GRUs are not inherently position-aware when processing irregular patterns. This combined sequence is passed through a two-layer GRU with dropout applied between the internal GRU layers. An additional dropout layer is applied to the final hidden state before decoding, as PyTorch's GRU dropout does not act on the output of the last layer.
The decoder consists of a two-layer feed-forward network with a LayerNorm applied after the first linear layer, followed by a ReLU activation and dropout. LayerNorm was preferred over BatchNorm for its robustness under distribution shift, which is a relevant concern in energy time series where the statistical properties of the signal can vary across seasons and years. The network outputs three quantile estimates: P10, P50, and P90, corresponding to a lower bound, median, and upper bound of the predicted production. Producing a forecast interval rather than a point estimate is particularly useful in an energy context, where understanding uncertainty is as important as the forecast itself. Quantile monotonicity (P10 ≤ P50 ≤ P90) is enforced via a differentiable sort operation, ensuring valid intervals without constraining the loss function.

// Spatio-Temporal GNN (STGNN) for Energy Resilience Diagram
// Render using Graphviz or a compatible viewer
digraph STGNN_Architecture {
    rankdir=TD; // Top to Down flow
    node [fontname="Helvetica,Arial,sans-serif", shape=box, style=filled, color="#E1E1E1", fontsize=10];
    edge [fontname="Helvetica,Arial,sans-serif", fontsize=9];

    // --- Graph Styles ---
    // Module Boxes
    subgraph cluster_style_layers { graph[style=invis]; node [fillcolor="#cfe2f3", color="#6fa8dc"]; } // Light Blue for Layers
    subgraph cluster_style_data { graph[style=invis]; node [fillcolor="#ffffff", color="#aaaaaa", shape=none, style=none]; } // White/Plain for Data
    subgraph cluster_style_op { graph[style=invis]; node [fillcolor="#fff2cc", color="#ffd966", shape=ellipse]; } // Yellow/Ellipse for Ops

    //=========================================
    // 1. INPUTS
    //=========================================
    subgraph cluster_0 {
        label = "1. Input Sequence (Graph Time Series)";
        style="filled,dashed"; color="#f3f3f3"; fontname="Helvetica-Bold"; fontsize=12;

        x_seq [label=<<b>Input Sequence x_seq</b><br/>(B, T, N, F)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];
        edge_index [label=<<b>edge_index</b><br/>(2, num_edges)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];
        edge_weight [label=<<b>edge_weight</b><br/>(num_edges,)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];
    }

    //=========================================
    // 2. SPATIO-TEMPORAL GNN BLOCK
    //=========================================
    subgraph cluster_1 {
        label = "2. Spatial Pass (Process Snapshots in Parallel)";
        style="filled"; color="#f1f1f1"; fontname="Helvetica-Bold"; fontsize=12;

        node [fillcolor="#cfe2f3", color="#6fa8dc"]; // Reset for layers
        
        // Flattening & Prepping
        reshape_1 [label=<<b>Flatten (B*T Snapshots)</b><br/>x_seq → x_flat<br/>N=4 (EE, FI, LV, LT)> shape=parallelogram];
        t_w_1 [label="Tile Edge Weights" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];

        // GAT Layer 1
        subgraph cluster_1a {
            label = "GAT Block 1"; style=filled; color="#cfe2f3"; fontname="Helvetica-Oblique";
            gat1 [label=<<b>GATv2Conv (gat1)</b><br/>In: F, Out: H<br/>Uses flow edge weights>];
            relu1 [label="ReLU" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];
            lin_res [label=<<b>Residual Projection (lin)</b><br/>Map F → H>];
            add1 [label="+" shape=circle fillcolor="#f3f3f3" color="#aaaaaa"];
        }

        // GAT Layer 2
        subgraph cluster_1b {
            label = "GAT Block 2"; style=filled; color="#cfe2f3"; fontname="Helvetica-Oblique";
            gat2 [label=<<b>GATv2Conv (gat2)</b><br/>In: H, Out: H<br/>Uses flow edge weights>];
            relu2 [label="ReLU" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];
            add2 [label="+" shape=circle fillcolor="#f3f3f3" color="#aaaaaa"];
        }

        // Output shaping
        reshape_2 [label=<<b>Reshape to Sequence</b><br/>→ (B, T, N, H)> shape=parallelogram];

        // Logical Flow inside Spatial Block
        x_seq -> reshape_1 [label="(B, T, N, F)"];
        reshape_1 -> lin_res;
        
        reshape_1 -> gat1 [label="(B*T*N, F)"];
        edge_index -> gat1;
        edge_weight -> t_w_1;
        t_w_1 -> gat1 [label="(B*T*E, 1)"];
        gat1 -> relu1;
        relu1 -> add1;
        lin_res -> add1;

        add1 -> gat2 [label="(B*T*N, H)"];
        edge_index -> gat2;
        t_w_1 -> gat2;
        gat2 -> relu2;
        add1 -> add2; // Residual connect previous hidden
        relu2 -> add2;
        add2 -> reshape_2 [label="(B*T*N, H)"];
    }

    //=========================================
    // 3. SPATIAL AGGREGATION & TEMPORAL PREP
    //=========================================
    subgraph cluster_2 {
        label = "3. Neighbor & Positional Context Prep";
        style="filled,dashed"; color="#fffaf0"; fontname="Helvetica-Bold"; fontsize=12;

        node [fillcolor="#ffffff", color="#aaaaaa", shape=none, style=none];

        // Slicing Nodes
        split_node [label="Split by Node Index" shape=diamond fillcolor="#fff2cc" color="#ffd966"];
        ee_features [label=<<b>EE Node features</b><br/>node 0, T steps<br/>(B, T, H)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];
        neigh_features [label=<<b>Neighbor features</b><br/>nodes [1:4] (FI,LV,LT), T steps<br/>(B, T, 3, H)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];

        // Weighted Aggregation
        attn_calc [label="Attention: MatMul & Softmax" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];
        apply_attn [label="Weighted Sum" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];
        neigh_agg [label=<<b>Aggregated Neighbors</b><br/>(B, T, H)> fillcolor="#eeeeee" shape=parallelogram style=filled];

        // Concatenation and Positional Embedding
        pos_emb [label=<<b>Positional Embedding (pos_emb)</b><br/>Lookup per timestep 0..T> fillcolor="#cfe2f3" color="#6fa8dc"];
        add_pos [label="+ (Add pos signal)" shape=circle fillcolor="#f3f3f3" color="#aaaaaa"];
        concat_struc [label=<<b>Concatenate Structure</b><br/>[EE_context, Neigh_context]> shape=parallelogram fillcolor="#eeeeee" style=filled];
        gru_input [label=<<b>GRU Input Sequence</b><br/>(B, T, 2*H)> fillcolor="#e6b8af" color="#dd7e6b" shape=parallelogram style=filled];

        // Connect flow
        reshape_2 -> split_node [label="(B, T, N, H)"];
        split_node -> ee_features;
        split_node -> neigh_features;

        ee_features -> attn_calc;
        neigh_features -> attn_calc [label="(transpose)"];
        attn_calc -> apply_attn [label="Weights (B,T,1,3)"];
        neigh_features -> apply_attn;
        apply_attn -> neigh_agg;

        ee_features -> add_pos;
        pos_emb -> add_pos;

        add_pos -> concat_struc [label="EE Context (B,T,H)"];
        neigh_agg -> concat_struc [label="Neigh Context (B,T,H)"];
        concat_struc -> gru_input;
    }

    //=========================================
    // 4. TEMPORAL ENCODING & DECODING
    //=========================================
    subgraph cluster_3 {
        label = "4. Temporal Gating & Quantile Decoding";
        style="filled"; color="#f1f1f1"; fontname="Helvetica-Bold"; fontsize=12;

        node [fillcolor="#d9ead3", color="#93c47d"]; // Green for Temporal/Output

        gru [label=<<b>GRU Layer (gru)</b><br/>2 layers, dropout=0.3<br/>Input: 2*H, Out: H>];
        last_state [label=<<b>Last Hidden State (h_n[-1])</b><br/>(B, H)> fillcolor="#eeeeee" shape=parallelogram style=filled color="#aaaaaa"];
        
        gru_drop [label=<<b>GRU Final Dropout (gru_dropout)</b><br/>rate=0.3>];

        // Decoder Stack
        subgraph cluster_3a {
            label = "Decoder (Sequential)"; style=filled; color="#d9ead3"; fontname="Helvetica-Oblique";
            dec_lin1 [label=<<b>Linear</b><br/>H → 2*H>];
            dec_ln [label=<<b>LayerNorm</b><br/>(not BatchNorm!)>];
            dec_relu [label="ReLU" shape=ellipse fillcolor="#fff2cc" color="#ffd966"];
            dec_drop [label=<<b>Dropout</b><br/>rate=0.3>];
            dec_lin2 [label=<<b>Linear</b><br/>2*H → 3 (Quantiles)>];
        }

        raw_out [label=<<b>Raw Predicted Quantiles</b><br/>(B, 3)> fillcolor="#eeeeee" shape=parallelogram style=filled color="#aaaaaa"];
        sort [label=<<b>torch.sort</b><br/>(Enforce P10 ≤ P50 ≤ P90)<br/>Differentiable> shape=ellipse fillcolor="#fff2cc" color="#ffd966"];

        // Output Parallelogram
        final_output [label=<<b>Final Prediction</b><br/>Ordered Quantiles<br/>(B, 3)> fillcolor="#fce5cd" color="#e06666" shape=parallelogram style="filled,bold"];

        // Connect flow
        gru_input -> gru;
        gru -> last_state;
        last_state -> gru_drop;
        gru_drop -> dec_lin1;
        dec_lin1 -> dec_ln;
        dec_ln -> dec_relu;
        dec_relu -> dec_drop;
        dec_drop -> dec_lin2;
        dec_lin2 -> raw_out;
        raw_out -> sort;
        sort -> final_output;
    }
}

### Training

The model was trained with the Adam optimiser for up to 50 epochs with early stopping (patience = 10): if validation loss does not improve for 10 consecutive epochs, training halts and the best checkpoint is restored.

### Scenarios

The trained model predicts Estonian domestic production for January 2026. Four scenarios are constructed as post-hoc adjustments to the model output, avoiding out-of-distribution extrapolation.

Since the model targets domestic production only, the full-grid scenario requires adding observed net cross-border flows back to the forecast. Isolation scenarios use the model output directly as the available energy, since no imports are possible. Wind scenarios add only the incremental generation from new capacity — computed as the difference between the scenario total and the baseline wind series — so that existing wind already captured in the model's training features is not double-counted.

- **S1 — Full grid (baseline):** net cross-border flows (imports and exports) are added to the model's production forecast, representing total available energy under the current interconnected grid.
- **S2 — Full isolation:** only domestic production is available, represented directly by the model output without any flow adjustment.
- **S3 — Isolation + Scenario A wind:** incremental generation from planned wind farm capacity is added to S2.
- **S4 — Isolation + Scenario B wind:** incremental generation from pipeline wind farm capacity is added to S2.

The P10–P90 uncertainty band from the model is shifted identically across all scenarios, reflecting that forecast uncertainty in domestic production applies regardless of the grid configuration.

# Detailed Description of Supply.py

## Overview

**Supply.py** is the main training and evaluation pipeline for an **Spatio-Temporal Graph Neural Network (ST-GNN)** that predicts Estonian electricity available energy (production + imports - exports) under various scenarios. The script:

1. Fetches 7 years of grid data (2019–2026) from Elering API and weather from Open-Meteo
2. Engineers 14 features capturing grid dynamics, weather, and calendar patterns
3. Creates sequences for supervised learning with explicit **lag24 feature engineering** to prevent target leakage
4. Trains a quantile-regression GNN to predict P10, P50, P90 energy supply forecasts
5. Evaluates on unseen January 2026 data (the actual energy crisis)
6. Runs counterfactual scenarios (full isolation, isolation + wind investment) with realistic wind production from weather-based modeling

## 1. Data Collection & Preprocessing (Lines 15–76)

### 1.1 API Fetching
```
START = "2019-01-01T00:00:00.000Z"
END   = "2026-02-01T00:00:00.000Z"
```

- **Prices**: NPS prices from Elering for EE, FI, LV, LT (15-min granularity)
- **Cross-border flows**: Power flows EE↔FI, EE↔LV, EE↔RU_Narva, EE↔RU_Pihkva (hourly)
- **System production**: Total EE production + renewable + frequency deviation (5-min to hourly)
- **Weather**: Temperature, wind speed at 10m from Open-Meteo for central Estonia (58.90°N, 24.75°E)

### 1.2 Resampling to Hourly
All data standardized to hourly frequency via `.resample("h").mean()`. This handles:
- 15-min price data → hourly average
- 5-min production data → hourly average
- Already-hourly flows → passed through for alignment

### 1.3 Handling Missing Data
```python
df_daily[flow_cols] = df_daily[flow_cols].fillna(0)  # Flows default to 0 (no trade)
df_daily = df_daily.dropna(how="all")  # Drop rows that are entirely missing
```

## 2. Feature Engineering & Energy Balance (Lines 82–189)

### 2.1 Available Energy (Target Definition)
```python
system_h["available_energy"] = (
    system_h["production"]
    - flows_h[("ee", "fi")]
    - flows_h[("ee", "lv")] 
    - flows_h[("ee", "ru_narva")]
    - flows_h[("ee", "ru_pihkva")]
)
```

**Key insight**: Elering API convention is:
- **Positive flow** = Estonia exports (reduces available energy)
- **Negative flow** = Estonia imports (increases available energy)

Available energy = domestic production − exports + imports = production − (all flows, with signs preserved)

Diagnostic checks (lines 113–119) verify the assumption that EE is a net importer from FI and LV.

### 2.2 Calendar Features (Cyclical Encoding)
```python
hour_sin = np.sin(2π * idx.hour / 24)      # Hour-of-day (0–23)
dow_sin = np.sin(2π * idx.dayofweek / 7)   # Day-of-week (0–6)
month_sin = np.sin(2π * (idx.month−1) / 12) # Month-of-year (0–11)
```

Sine/cosine encoding prevents treating Mon=1, Sun=7 as numeric distance. Captures diurnal and seasonal patterns.

### 2.3 Grid Stress Signal
```python
freq_deviation = system_h["frequency"] - 50.0
```

Frequency drops below 50 Hz when supply is tight (high demand or low production). Real-time stress indicator.

### 2.4 Wind Feature
```python
wind_mw = entsoe["wind_onshore"].reindex(idx, method="ffill").fillna(0)
```

ENTSOE actual wind production data. Already included in `production`, but tracked separately so the model can learn wind-specific patterns and inject counterfactual scenarios.

### 2.5 Feature List (14 total per node)
**EE node (full data)**:
- `available_energy_lag24` ← **Lagged by HORIZON (24h) to prevent target leakage**
- `production_renewable`, `production`, `wind_mw` (energy sources)
- `flow_fi`, `flow_lv` (cross-border trade)
- `price` (market signal)
- `temperature`, `wind_speed_10m` (weather)
- `freq_deviation` (grid stress)
- `hour_sin`, `hour_cos`, `dow_sin`, `dow_cos`, `month_sin`, `month_cos` (calendar)

**FI/LV nodes (partial data, zero-padded for missing features)**:
- `price`, `flow_*` (neighboring countries' market + trade)
- All other features = 0

**LT node**:
- `price` only
- All other features = 0

### 2.6 Lag24 Leakage Prevention (Critical Design)
```python
"available_energy_lag24": system_h["available_energy"].shift(HORIZON)
```

**Problem**: If the model saw current available energy, it could "copy" recent values that overlap with the target window (24h ahead). Circular logic.

**Solution**: Shift available energy by 24h. The model only sees energy balance from before the prediction horizon:
- Input: available energy from hours [t−48, t−24)
- Target: available energy at hour t+23

No overlap → no leakage, but genuine predictive signal is preserved.

## 3. Graph Structure (Lines 241–245)

```python
edge_index = torch.tensor([
    [0, 1, 0, 2, 2, 3],  # source nodes
    [1, 0, 2, 0, 3, 2],  # target nodes (bidirectional)
], dtype=torch.long)
```

**Topology**: EE (node 0) at center:
- EE ↔ FI (nodes 0–1, bidirectional)
- EE ↔ LV (nodes 0–2, bidirectional)
- LV ↔ LT (nodes 2–3, bidirectional)

Models actual interconnections. Russia intentionally excluded (poor feature coverage).

## 4. Node Data Stacking (Lines 169–174)

```python
node_data = np.stack([
    ee_feats.values,   # node 0: EE (14 features)
    fi_feats.values,   # node 1: FI (14 features, mostly zeros)
    lv_feats.values,   # node 2: LV (14 features, mostly zeros)
    lt_feats.values,   # node 3: LT (14 features, mostly zeros)
], axis=1)
```

Shape: **(T, NUM_NODES, NUM_FEATURES)** = (60264 hours, 4 nodes, 14 features)

Ready for sequence creation.

## 5. Sequence Creation & Temporal Split (Lines 196–224)

### 5.1 Sliding Window Sequences
```python
SEQ_LEN = 48   # 2 days of history
HORIZON = 24   # predict 24 hours ahead

for t in range(SEQ_LEN, len(node_data) - HORIZON):
    X_list.append(node_data[t - SEQ_LEN:t])         # Input: [t-48, t)
    y_list.append(y_target_values[t + HORIZON - 1]) # Target: t+23
```

Each sample:
- **Input X**: 48 timesteps of (4 nodes, 14 features)
- **Target y**: scalar (available energy at t+23)

Total: ~60,000 sequences

### 5.2 Temporal Train/Val/Test Split
```python
jan2026_start = np.searchsorted(idx[SEQ_LEN:-HORIZON], pd.Timestamp("2026-01-01"))

X_train_full, y_train_full = X[:jan2026_start], y[:jan2026_start]
X_test, y_test = X[jan2026_start:], y[jan2026_start:]

val_split = int(len(X_train_full) * 0.8)
X_train, y_train = X_train_full[:val_split], y_train_full[:val_split]
X_val, y_val = X_train_full[val_split:], y_train_full[val_split:]
```

**Why temporal split?** Real-world forecasting: train on past, test on future. No information leakage.

- **Train**: 2019–Oct 2025 (~56k sequences)
- **Val**: Oct 2025–Dec 2025 (~5k sequences)
- **Test**: January 2026 (crisis month, ~744 sequences) — **held-out unseen data**

## 6. Normalization (Lines 227–238)

```python
scaler = ps.PowerScaler(X_train, y_train)  # Fit ONLY on training data

X_train_t = scaler.scale_x(X_train)
X_val_t = scaler.scale_x(X_val)
X_test_t = scaler.scale_x(X_test)

y_train_t = scaler.scale_y(y_train).view(-1, 1)
y_val_t = scaler.scale_y(y_val).view(-1, 1)
y_test_t = scaler.scale_y(y_test).view(-1, 1)
```

**PowerScaler** (custom normalization class):
- Fits mean and std on training data only
- Applies same transformation to val/test
- Prevents leakage: val/test stats don't influence training

Each feature independently standardized: `(x - mean) / std`

## 7. Model Initialization (Lines 248–257)

```python
model = STGNN(NUM_FEATURES=14, hidden_dim=64).to(device)
edge_index = edge_index.to(device)
x_mean = scaler.x_mean.to(device)
x_std = scaler.x_std.to(device)
```

**STGNN** architecture:
- **Input**: (Batch, SEQ_LEN, NUM_NODES, NUM_FEATURES) = (B, 48, 4, 14)
- **Hidden dimension**: 64 (increased from 32 for better capacity)
- **Output**: (Batch, 3) → [P10, P50, P90] quantiles via quantile loss
- **Mechanism**: Temporal convolution + graph attention/message passing

Device: CUDA → MPS (Apple Silicon) → CPU fallback

## 8. Training Loop (Lines 268–342)

### 8.1 Hyperparameters
```python
optimizer = torch.optim.Adam(lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(patience=5, factor=0.5)
early_stopping = opt.EarlyStopping(patience=15, min_delta=1e-4)

BATCH_SIZE = 64
EPOCHS = 200  # upper bound
```

- **Learning rate**: 5e-4 (lowered from 1e-3 for stability)
- **Weight decay**: 1e-3 (L2 regularization)
- **Scheduler**: Reduce LR by 0.5× if val loss plateaus for 5 epochs
- **Early stopping**: Stop if val loss doesn't improve by 1e-4 for 15 epochs

### 8.2 Per-Epoch Training
```python
for epoch in range(EPOCHS):
    model.train()
    for batch indices:
        xb, yb = batch
        preds = model(xb, edge_index)
        loss = helper.quantile_loss(preds, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
```

**Quantile loss**: Custom loss that minimizes prediction error for P10, P50, P90 simultaneously. Ensures well-calibrated uncertainty estimates.

**Gradient clipping**: Prevent exploding gradients (norm ≤ 1.0).

### 8.3 Validation & Early Stopping
```python
model.eval()
for val_batch:
    v_loss = helper.quantile_loss(model(xb_v, edge_index), yb_v)
    val_losses_b.append(v_loss.item())

avg_val_loss = np.mean(val_losses_b)
scheduler.step(avg_val_loss)
early_stopping.step(avg_val_loss, model, epoch)
if early_stopping.stop:
    break
```

Checkpoints best model (lowest val loss) and restores before evaluation.

## 9. Evaluation on January 2026 (Lines 348–382)

```python
model.eval()
with torch.no_grad():
    X_test_device = X_test_t.to(device)
    test_preds = model(X_test_device, edge_index).cpu().numpy()  # Shape: (744, 3)

p10 = scaler.inverse_y(test_preds[:, 0])
p50 = scaler.inverse_y(test_preds[:, 1])
p90 = scaler.inverse_y(test_preds[:, 2])
actual = scaler.inverse_y(y_test_t.squeeze().numpy())
```

**Metrics**:
- **MAE**: Mean absolute error between P50 and actual
- **RMSE**: Root mean square error
- **P10–P90 coverage**: Fraction of actual values within prediction interval (target ≥ 80%)
- **Mean/min/max supply**: Forecast statistics

## 10. Scenario Engine (Lines 386–456)

### 10.1 Function Signature
```python
def run_scenario(name, isolate=False, wind_series=None, extra_wind_mw=0):
```

- **isolate**: Remove cross-border edges (Estonia fully island mode)
- **wind_series**: Np.array of counterfactual wind production (48 values per sequence)
- **extra_wind_mw**: Flat MW capacity boost

### 10.2 Isolation Scenario (S2)
```python
if isolate:
    edges = torch.zeros((2, 0), dtype=torch.long)  # Empty edge list
    
    fi_flow_orig = x_mod[:, :, 0, FLOW_FI_IDX].clone()
    lv_flow_orig = x_mod[:, :, 0, FLOW_LV_IDX].clone()
    
    x_mod[:, :, 0, FLOW_FI_IDX] = 0.0
    x_mod[:, :, 1, FLOW_FI_IDX] = 0.0
    x_mod[:, :, 0, FLOW_LV_IDX] = 0.0
    x_mod[:, :, 2, FLOW_LV_IDX] = 0.0
    
    x_mod[:, :, 0, SUPPLY_IDX] += fi_flow_orig + lv_flow_orig
```

**Logic**:
1. Zero out all cross-border flows (EE can't trade)
2. Save original flow values (negative = imports)
3. Add imports back to available energy (convert from flows to supply)
4. Result: EE can only use domestic production

### 10.3 Wind Scenario Injection
```python
if wind_series is not None:
    renew_std = x_std[0, 0, 0, RENEW_IDX].item()
    prod_std = x_std[0, 0, 0, PROD_IDX].item()
    supply_std = x_std[0, 0, 0, SUPPLY_IDX].item()
    wind_mw_std = x_std[0, 0, 0, WIND_MW_IDX].item()
    
    for t in range(len(x_mod)):
        w = wind_series[t : t + SEQ_LEN]
        if len(w) == SEQ_LEN:
            wt = torch.from_numpy(w.astype(np.float32))
            x_mod[t, :, 0, RENEW_IDX] += wt / renew_std
            x_mod[t, :, 0, PROD_IDX] += wt / prod_std
            x_mod[t, :, 0, SUPPLY_IDX] += wt / supply_std
            x_mod[t, :, 0, WIND_MW_IDX] += wt / wind_mw_std
```

**Key points**:
- **wind_series**: Counterfactual wind delta (already has baseline subtracted, e.g., "+323 MW new")
- **For each sequence t**: Take 48 hours of wind data (aligns with input history)
- **Normalize independently**: Each feature divided by its own training std (no ratio scaling)
- **Update 4 features**: RENEW, PROD, SUPPLY, WIND_MW all affected by wind injection
- **Why 4 features?** Wind affects renewable source (RENEW), total production (PROD), available supply (SUPPLY), and is tracked separately (WIND_MW)

### 10.4 Scenario Execution
```python
p50_s1, p10_s1, p90_s1 = run_scenario("S1: Full grid — all connections intact")
p50_s2, p10_s2, p90_s2 = run_scenario("S2: Full isolation", isolate=True)
p50_s3, p10_s3, p90_s3 = run_scenario(
    "S3: Isolated + Scenario A", isolate=True, wind_series=wind_scenA)
p50_s4, p10_s4, p90_s4 = run_scenario(
    "S4: Isolated + Scenario B", isolate=True, wind_series=wind_scenB)
```

## 11. Wind Production Scenarios (Lines 459–473)

```python
wind_scenarios = pd.read_csv("../data/wind_production_scenarios.csv")
baseline = wind_scenarios["wind_mwh_baseline"].values
wind_scenA = (
    wind_scenarios["wind_mwh_scenA"] - wind_scenarios["wind_mwh_baseline"]
).values  # +323 MW new
wind_scenB = (
    wind_scenarios["wind_mwh_scenB"] - wind_scenarios["wind_mwh_baseline"]
).values  # +887 MW new
```

CSV generated by `ursula_wind_counterfactual.ipynb`:
- **Baseline**: January 2026 actual wind (50% below historical average)
- **Scenario A**: Add wind farms (Lääneranna, Pärnu, Aidu) = +323 MW
- **Scenario B**: Add all pipeline farms (5 municipalities) = +887 MW total new capacity

Subtraction ensures we only inject the *additional* wind, not double-count baseline.

## 12. Visualization (Lines 510–595)

### 12.1 Plot 1: Training Curves
```python
axes[0, 0].plot(train_losses, label="Train")
axes[0, 0].plot(val_losses, label="Val")
```

Shows train/val loss over epochs. Indicates overfitting (divergence) or early stopping trigger.

### 12.2 Plot 2: Forecast vs Actual (S1)
```python
axes[0, 1].fill_between(jan_hours, p10, p90, alpha=0.15, label="P10–P90 band")
axes[0, 1].plot(jan_hours, p50, label="P50 forecast")
axes[0, 1].plot(jan_hours, actual, linestyle="--", label="Actual balance")
```

Calibration check: Does actual fall within prediction interval 80%+ of the time?

### 12.3 Plot 3: Scenario Comparison Over Time
```python
axes[1, 0].plot(jan_hours, p50_s1, label="Baseline")
axes[1, 0].plot(jan_hours, p50_s2, label="Isolated (no wind)")
axes[1, 0].plot(jan_hours, p50_s3, label="Scenario A (established plans)")
axes[1, 0].plot(jan_hours, p50_s4, label="Scenario B (pipeline farms)")
axes[1, 0].fill_between(jan_hours, p50_s1, p50_s2, alpha=0.15, label="Import dependency gap")
```

Shows resilience impact of wind investments.

### 12.4 Plot 4: Mean & Worst Supply by Scenario
```python
axes[1, 1].bar(x, mean_supply, color=colors, label="Mean available supply (P50)")
axes[1, 1].scatter(x, min_supply, marker="v", label="Worst hour (P10)")
axes[1, 1].axhline(900, linestyle="--", label="Typical consumption (~900 MW)")
```

Summary statistics: Which scenarios can meet typical 900 MW demand?

## Summary of Data Flow

```
Elering API + Open-Meteo Weather
    ↓
Resample to hourly, merge flows/prices/production/weather
    ↓
Engineer 14 features per node (lag24, calendar, stress signals)
    ↓
Stack into (T, 4 nodes, 14 features)
    ↓
Create 48-hour input / 24-hour target sequences
    ↓
Temporal split: Train (2019–Oct 2025) | Val (Oct–Dec 2025) | Test (Jan 2026)
    ↓
Normalize via PowerScaler (fit on train only)
    ↓
Train ST-GNN with quantile loss, early stopping, gradient clipping
    ↓
Evaluate on unseen Jan 2026 (744 hours)
    ↓
Run 4 scenarios: baseline, isolation, isolation+wind A, isolation+wind B
    ↓
Visualize: training curves, calibration, scenario comparison, summary stats
```

## Key Design Decisions

| Decision | Rationale |
|----------|----------|
| **lag24 feature** | Prevents target leakage while preserving predictive signal |
| **Temporal split** | Simulates real-world forecasting; no future-peeking |
| **Quantile regression** | Risk-aware resilience evaluation (P10 worst-case, P50 mean, P90 best) |
| **4-node graph** | Models actual interconnections; Russia excluded due to poor data |
| **Independent feature normalization** | Correct scaling when injecting counterfactual wind |
| **Baseline wind subtraction** | Prevents double-counting existing wind in scenarios |
| **Early stopping + scheduler** | Avoids overfitting; reduces LR on plateau |